# Description
process pcap data for get traditional baseline
label.pcap -> raw hex code
把原始的pcap流量文件处理成给传统机器学习baseline用的表格特征

In [2]:
import os
import logging
import scapy.all as scapy
from multiprocessing import Pool, cpu_count
import csv
import pandas as pd

os.chdir('/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML')
os.makedirs('logs', exist_ok=True)

logging.basicConfig(       
    level=logging.INFO,            
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',  
    handlers=[
        logging.FileHandler('logs/raw.log', mode='w'),  
        logging.StreamHandler()          
    ],
    force=True
)

logger = logging.getLogger()

In [3]:
dataset_path = '/home/a/traffic_encryption/debunk_data/Debunk_Traffic_Representation/packet-level-classification/per-flow-split/ustc-binary'
output_path = '/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/outputs/ustc-binary'

os.makedirs('logs', exist_ok=True)
os.makedirs(output_path, exist_ok=True)

把包里的 容易泄露标签的信息去掉 保留适合做baseline的内容 数据清洗
体现了论文的第四个data cleaning的部分 Filters removing header information

In [4]:
def clean_packet(packet):
    if packet.haslayer(scapy.Ether):
        packet = packet[scapy.Ether].payload#只保留网络层和传输层

    if packet.haslayer(scapy.IP):
        packet[scapy.IP].src = "0.0.0.0"
        packet[scapy.IP].dst = "0.0.0.0"
    elif packet.haslayer('IPv6'):
        packet['IPv6'].src = "::"
        packet['IPv6'].dst = "::"
    else:
        return 0

    if packet.haslayer(scapy.UDP):
        packet[scapy.UDP].sport = 0  # 设置源端口为0
        packet[scapy.UDP].dport = 0  # 设置目的端口为0
        packet[scapy.UDP].remove_payload()  # 删除 UDP 载荷
    elif packet.haslayer(scapy.TCP):
        packet[scapy.TCP].sport = 0  # 设置源端口为0
        packet[scapy.TCP].dport = 0  # 设置目的端口为0
        packet[scapy.TCP].remove_payload()  # 删除 TCP 载荷
    
    return packet

In [5]:
# get how many features/ip flags/tcp flags/.. in the dataset
# # file/{test.pcap} or file/{train_val_split_0}/{train.pcap}
max_payload_length = 0

for type in os.listdir(dataset_path):
    logger.info(f'Processing {type}')

#遍历ustc-binary根目录下的 test  train_val_split_012

    if 'test' in type:
        if 'pcap' in type:
            packets =  scapy.PcapReader(f'{dataset_path}/{type}')
            for id, packet in enumerate(packets):
                # only require ip header and transportaion layer
                packet = clean_packet(packet)
                if packet == 0:
                    continue

                hex_str = bytes(packet).hex()
                if len(hex_str) > max_payload_length:
                    max_payload_length = len(hex_str)
        continue

    for class_id, file_name in enumerate(os.listdir(f'{dataset_path}/{type}')):
        logger.info(f'Processing {type} {file_name}')
        if 'pcap' not in file_name:
            continue

        packets =  scapy.PcapReader(f'{dataset_path}/{type}/{file_name}')
        for id, packet in enumerate(packets):
            # only require ip header and transportaion layer
            packet = clean_packet(packet)
            if packet == 0:
                continue

            hex_str = bytes(packet).hex()
            if len(hex_str) > max_payload_length:
                max_payload_length = len(hex_str)
            
print(max_payload_length)


2026-04-22 10:16:08,535 - root - INFO - Processing train_val_split_0
2026-04-22 10:16:08,536 - root - INFO - Processing train_val_split_0 train
2026-04-22 10:16:08,536 - root - INFO - Processing train_val_split_0 val
2026-04-22 10:16:08,536 - root - INFO - Processing test
2026-04-22 10:16:08,537 - root - INFO - Processing train_val_split_2
2026-04-22 10:16:08,537 - root - INFO - Processing train_val_split_2 train
2026-04-22 10:16:08,538 - root - INFO - Processing train_val_split_2 val
2026-04-22 10:16:08,538 - root - INFO - Processing train_val_split_1
2026-04-22 10:16:08,539 - root - INFO - Processing train_val_split_1 train
2026-04-22 10:16:08,539 - root - INFO - Processing train_val_split_1 val


0


In [6]:
# get how many features/ip flags/tcp flags/.. in the dataset
# # file/test/{test.pcap} or file/train_val_split_0/{train/val}/.pcap
#摸清数据集里有哪些字段和特征 
max_payload_length = 0

for type in os.listdir(dataset_path):
    logger.info(f'Processing {type}')

    if 'test' == type:
        for class_id, file_name in enumerate(os.listdir(f'{dataset_path}/{type}')):
            if 'pcap' in file_name:
                logger.info(f'Processing {type} {file_name}')

                packets =  scapy.PcapReader(f'{dataset_path}/{type}/{file_name}')
                for id, packet in enumerate(packets):
                    # only require ip header and transportaion layer
                    packet = clean_packet(packet)
                    if packet == 0:
                        continue

                    hex_str = bytes(packet).hex()#把清洗后的packet转成十六进制字符串 传统模型没法直接读packet 需要表示成一个固定 离散 可切片的形式
                    if len(hex_str) > max_payload_length:
                        max_payload_length = len(hex_str)
    else:
        for class_id, folder in enumerate(os.listdir(f'{dataset_path}/{type}')):
            logger.info(f'Processing {type} {folder}')

            for file_name in os.listdir(f'{dataset_path}/{type}/{folder}'):
                if 'pcap' in file_name:
                    packets =  scapy.PcapReader(f'{dataset_path}/{type}/{folder}/{file_name}')
                    for id, packet in enumerate(packets):
                        # only require ip header and transportaion layer
                        packet = clean_packet(packet)
                        if packet == 0:
                            continue

                        hex_str = bytes(packet).hex()
                        if len(hex_str) > max_payload_length:
                            max_payload_length = len(hex_str)
            
print(max_payload_length)

2026-04-22 10:16:08,564 - root - INFO - Processing train_val_split_0
2026-04-22 10:16:08,565 - root - INFO - Processing train_val_split_0 train
2026-04-22 10:16:33,066 - root - INFO - Processing train_val_split_0 val
2026-04-22 10:16:34,124 - root - INFO - Processing test
2026-04-22 10:16:34,125 - root - INFO - Processing test benign.pcap
2026-04-22 10:18:12,142 - root - INFO - Processing test malware.pcap
2026-04-22 10:19:09,544 - root - INFO - Processing train_val_split_2
2026-04-22 10:19:09,545 - root - INFO - Processing train_val_split_2 train
2026-04-22 10:19:37,950 - root - INFO - Processing train_val_split_2 val
2026-04-22 10:19:39,264 - root - INFO - Processing train_val_split_1
2026-04-22 10:19:39,265 - root - INFO - Processing train_val_split_1 train
2026-04-22 10:20:05,847 - root - INFO - Processing train_val_split_1 val


136


In [7]:
# # file/{test.pcap} or file/{train_val_split_0}/{train.pcap}
byte_vocab = set()

for type in os.listdir(dataset_path):
    logger.info(f'Processing {type}')

    if 'test' in type:
        if 'pcap' in type:
            packets =  scapy.PcapReader(f'{dataset_path}/{type}')
            for id, packet in enumerate(packets):
                # only require ip header and transportaion layer
                packet = clean_packet(packet)
                if packet == 0:
                    continue

                hex_str = bytes(packet).hex()
                if len(hex_str) < max_payload_length:
                    hex_str = hex_str + '0' * (max_payload_length - len(hex_str))#padding到统一长度
                hex_array = [hex_str[i:i+4] for i in range(0, len(hex_str), 4)]#2个hex=1byte 4个hex=2byte

                byte_vocab.update(hex_array)#把包里出现过的2-byte token 加入全局词表集合 出现过136个😂
        continue

    for class_id, file_name in enumerate(os.listdir(f'{dataset_path}/{type}')):
        logger.info(f'Processing {type} {file_name}')
        if 'pcap' not in file_name:
            continue

        packets =  scapy.PcapReader(f'{dataset_path}/{type}/{file_name}')
        for id, packet in enumerate(packets):
            # only require ip header and transportaion layer
            packet = clean_packet(packet)
            if packet == 0:
                continue

            hex_str = bytes(packet).hex()
            if len(hex_str) < max_payload_length:
                hex_str = hex_str + '0' * (max_payload_length - len(hex_str))
            hex_array = [hex_str[i:i+4] for i in range(0, len(hex_str), 4)]

            byte_vocab.update(hex_array)

byte_list = list(byte_vocab)
print(len(byte_list))  

2026-04-22 10:20:06,793 - root - INFO - Processing train_val_split_0
2026-04-22 10:20:06,794 - root - INFO - Processing train_val_split_0 train
2026-04-22 10:20:06,795 - root - INFO - Processing train_val_split_0 val
2026-04-22 10:20:06,795 - root - INFO - Processing test
2026-04-22 10:20:06,796 - root - INFO - Processing train_val_split_2
2026-04-22 10:20:06,796 - root - INFO - Processing train_val_split_2 train
2026-04-22 10:20:06,797 - root - INFO - Processing train_val_split_2 val
2026-04-22 10:20:06,797 - root - INFO - Processing train_val_split_1
2026-04-22 10:20:06,798 - root - INFO - Processing train_val_split_1 train
2026-04-22 10:20:06,799 - root - INFO - Processing train_val_split_1 val


0


In [8]:
# # file/test/{test.pcap} or file/train_val_split_0/{train/val}/.pcap
byte_vocab = set()

for type in os.listdir(dataset_path):
    logger.info(f'Processing {type}')

    if 'test' == type:
        for class_id, file_name in enumerate(os.listdir(f'{dataset_path}/{type}')):
            if 'pcap' in file_name:
                logger.info(f'Processing {type} {file_name}')

                packets =  scapy.PcapReader(f'{dataset_path}/{type}/{file_name}')
                for id, packet in enumerate(packets):
                    # only require ip header and transportaion layer
                    packet = clean_packet(packet)
                    if packet == 0:
                        continue

                    hex_str = bytes(packet).hex()
                    if len(hex_str) < max_payload_length:
                        hex_str = hex_str + '0' * (max_payload_length - len(hex_str))
                    hex_array = [hex_str[i:i+4] for i in range(0, len(hex_str), 4)]

                    byte_vocab.update(hex_array)
    else:
        for class_id, folder in enumerate(os.listdir(f'{dataset_path}/{type}')):
            logger.info(f'Processing {type} {folder}')

            for file_name in os.listdir(f'{dataset_path}/{type}/{folder}'):
                if 'pcap' in file_name:
                    packets =  scapy.PcapReader(f'{dataset_path}/{type}/{folder}/{file_name}')
                    for id, packet in enumerate(packets):
                        # only require ip header and transportaion layer
                        packet = clean_packet(packet)
                        if packet == 0:
                            continue

                        hex_str = bytes(packet).hex()
                        if len(hex_str) < max_payload_length:
                            hex_str = hex_str + '0' * (max_payload_length - len(hex_str))
                        hex_array = [hex_str[i:i+4] for i in range(0, len(hex_str), 4)]

                        byte_vocab.update(hex_array)

byte_list = list(byte_vocab)
print(len(byte_list))  

2026-04-22 10:20:06,827 - root - INFO - Processing train_val_split_0
2026-04-22 10:20:06,828 - root - INFO - Processing train_val_split_0 train
2026-04-22 10:20:34,740 - root - INFO - Processing train_val_split_0 val
2026-04-22 10:20:37,985 - root - INFO - Processing test
2026-04-22 10:20:37,986 - root - INFO - Processing test benign.pcap
2026-04-22 10:22:15,397 - root - INFO - Processing test malware.pcap
2026-04-22 10:23:15,297 - root - INFO - Processing train_val_split_2
2026-04-22 10:23:15,298 - root - INFO - Processing train_val_split_2 train
2026-04-22 10:23:40,311 - root - INFO - Processing train_val_split_2 val
2026-04-22 10:23:43,613 - root - INFO - Processing train_val_split_1
2026-04-22 10:23:43,614 - root - INFO - Processing train_val_split_1 train
2026-04-22 10:24:09,326 - root - INFO - Processing train_val_split_1 val


65536


In [9]:
# # file/{test.pcap} or file/{train_val_split_0}/{train.pcap}
def write2csv(type, data, label):
    file_path = f'{output_path}/raw/{type}.csv'

    if not os.path.isfile(file_path):
        with open(file_path, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(list([str(i) for i in range(int(max_payload_length/4))]) + ['class'])

    with open(file_path, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(data + [label])


for type in os.listdir(dataset_path):
    logger.info(f'Processing {type}')
    if 'test' in type: 
        if 'pcap' in type:
            packets = scapy.PcapReader(f'{dataset_path}/{type}')
            data_frame = pd.read_parquet(f'{dataset_path}/{type[:-5]}.parquet')

            def process_packet(packet, id):
                packet = clean_packet(packet)
                if packet == 0:
                    return None
                
                hex_str = bytes(packet).hex()
                if len(hex_str) < max_payload_length:
                    hex_str = hex_str + '0' * (max_payload_length - len(hex_str))
                hex_array = [byte_list.index(hex_str[i:i+4]) for i in range(0, len(hex_str), 4)]
                return (hex_array, data_frame.iloc[id]['class_str'])

            with Pool(cpu_count()) as pool:
                results = pool.starmap(process_packet, [(packet, id) for id, packet in enumerate(packets)])
                
            for result in results:
                if result is not None:
                    write2csv(type[:-5], result[0], result[1])
    else:
        for class_id, file_name in enumerate(os.listdir(f'{dataset_path}/{type}')):
            logger.info(f'Processing {type} {file_name}')
            if 'pcap' in file_name:
                packets =  scapy.PcapReader(f'{dataset_path}/{type}/{file_name}')
                data_frame = pd.read_parquet(f'{dataset_path}/{type}/{file_name[:-5]}.parquet')
                
                def process_packet(packet, id):
                    packet = clean_packet(packet)
                    if packet == 0:
                        return None
                    
                    hex_str = bytes(packet).hex()
                    if len(hex_str) < max_payload_length:
                        hex_str = hex_str + '0' * (max_payload_length - len(hex_str))
                    hex_array = [byte_list.index(hex_str[i:i+4]) for i in range(0, len(hex_str), 4)]
                    return (hex_array, data_frame.iloc[id]['class_str'])

                with Pool(cpu_count()) as pool:
                    results = pool.starmap(process_packet, [(packet, id) for id, packet in enumerate(packets)])
                    
                for result in results:
                    if result is not None:
                        write2csv(f'{type}/{file_name[:-5]}', result[0], result[1])
        
logger.info('Finished')  

2026-04-22 10:24:12,632 - root - INFO - Processing train_val_split_0
2026-04-22 10:24:12,633 - root - INFO - Processing train_val_split_0 train
2026-04-22 10:24:12,633 - root - INFO - Processing train_val_split_0 val
2026-04-22 10:24:12,633 - root - INFO - Processing test
2026-04-22 10:24:12,634 - root - INFO - Processing train_val_split_2
2026-04-22 10:24:12,635 - root - INFO - Processing train_val_split_2 train
2026-04-22 10:24:12,635 - root - INFO - Processing train_val_split_2 val
2026-04-22 10:24:12,635 - root - INFO - Processing train_val_split_1
2026-04-22 10:24:12,636 - root - INFO - Processing train_val_split_1 train
2026-04-22 10:24:12,636 - root - INFO - Processing train_val_split_1 val
2026-04-22 10:24:12,637 - root - INFO - Finished


In [ ]:
# # file/test/{test.pcap} or file/train_val_split_0/{train/val}/.pcap
import shutil

raw_output_path = os.path.join(output_path, 'raw')
if os.path.isdir(raw_output_path):
    logger.info(f'Removing existing raw output directory: {raw_output_path}')
    shutil.rmtree(raw_output_path)
os.makedirs(raw_output_path, exist_ok=True)

def write2csv(type, data, label):
    file_path = os.path.join(raw_output_path, f'{type}.csv')
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    if not os.path.isfile(file_path):
        with open(file_path, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(list([str(i) for i in range(int(max_payload_length/4))]) + ['class'])

    with open(file_path, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(data + [label])


for type in os.listdir(dataset_path):
    logger.info(f'Processing {type}')

    if 'test' == type:
        for class_id, file_name in enumerate(os.listdir(f'{dataset_path}/{type}')):
            if 'pcap' in file_name:
                logger.info(f'Processing {type} {file_name}')

                def process_packet(packet, id):
                    packet = clean_packet(packet)
                    if packet == 0:
                        return None
                    
                    hex_str = bytes(packet).hex() #把清洗后的包转换成十六进制字符串
                    if len(hex_str) < max_payload_length:#补零统一长度
                        hex_str = hex_str + '0' * (max_payload_length - len(hex_str))
                    hex_array = [byte_list.index(hex_str[i:i+4]) for i in range(0, len(hex_str), 4)]#hex array是一条packet的固定长度整数特征向量
                    return (hex_array, file_name)

                packets =  scapy.PcapReader(f'{dataset_path}/{type}/{file_name}')
                with Pool(cpu_count()) as pool:
                    results = pool.starmap(process_packet, [(packet, id) for id, packet in enumerate(packets)])
                    
                for result in results:
                    if result is not None:
                        write2csv(type, result[0], result[1])
    else:
        for class_id, folder in enumerate(os.listdir(f'{dataset_path}/{type}')):
            logger.info(f'Processing {type} {folder}')

            for file_name in os.listdir(f'{dataset_path}/{type}/{folder}'):
                if 'pcap' in file_name:
                    def process_packet(packet, id):
                        packet = clean_packet(packet)
                        if packet == 0:
                            return None
                        
                        hex_str = bytes(packet).hex()
                        if len(hex_str) < max_payload_length:
                            hex_str = hex_str + '0' * (max_payload_length - len(hex_str))
                        hex_array = [byte_list.index(hex_str[i:i+4]) for i in range(0, len(hex_str), 4)]
                        return (hex_array, file_name)
                    
                    packets =  scapy.PcapReader(f'{dataset_path}/{type}/{folder}/{file_name}')
                    with Pool(cpu_count()) as pool:
                        results = pool.starmap(process_packet, [(packet, id) for id, packet in enumerate(packets)])
                        
                    for result in results:
                        if result is not None:
                            write2csv(f'{type}/{folder}', result[0], result[1])
        
logger.info('Finished')  

2026-04-22 10:24:12,660 - root - INFO - Processing train_val_split_0
2026-04-22 10:24:12,661 - root - INFO - Processing train_val_split_0 train


2026-04-22 10:32:24,880 - root - INFO - Processing train_val_split_0 val
2026-04-22 10:33:38,879 - root - INFO - Processing test
2026-04-22 10:33:38,880 - root - INFO - Processing test benign.pcap
2026-04-22 11:09:48,282 - root - INFO - Processing test malware.pcap
2026-04-22 11:33:56,888 - root - INFO - Processing train_val_split_2
2026-04-22 11:33:56,951 - root - INFO - Processing train_val_split_2 train
2026-04-22 11:44:11,756 - root - INFO - Processing train_val_split_2 val
2026-04-22 11:45:29,282 - root - INFO - Processing train_val_split_1
2026-04-22 11:45:29,282 - root - INFO - Processing train_val_split_1 train
2026-04-22 11:55:32,244 - root - INFO - Processing train_val_split_1 val
2026-04-22 11:56:51,349 - root - INFO - Finished


## raw.ipynb：从 pcap 到 csv

### 这一步的作用
把原始 pcap 转换成 shallow baseline 可以使用的 csv 特征。

### 主要流程
1. 读取 pcap  
2. 调用 `clean_packet()` 清洗包  
3. 转成十六进制字符串 `hex_str`  
4. 按 4 个 hex 字符切成 token  
5. 映射为 `hex_array`  
6. 写入 csv，最后一列是 `class`

### 关键变量
- `dataset_path`：输入数据目录
- `output_path`：输出目录
- `hex_str`：packet 的十六进制表示
- `hex_array`：最终特征序列

### 我自己的理解
这一步不是训练模型，而是在做特征工程和数据准备。